In [ ]:
# Cell 1: Setup Environment & Load Model
# Fokus pada inisialisasi library dan model AI.

import re
import pandas as pd
import numpy as np
# import matplotlib.pyplot as plt
# import seaborn as sns
from nltk import ngrams
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer
from keybert import KeyBERT
from sklearn.metrics.pairwise import cosine_similarity

# Setup tema visualisasi
# sns.set_theme(style="whitegrid", palette="muted")

print("Sedang memuat Model Semantik (SBERT & KeyBERT)...")
model_sbert = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')
kw_model = KeyBERT(model=model_sbert) #type:ignore
print("Model Semantik siap digunakan!")

Sedang memuat Model Semantik (SBERT & KeyBERT)...


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 3462.83it/s]


Model Semantik siap digunakan!


In [14]:
# Cell 2: Kamus Ekspansi & Stopwords
# kamus ekspansi dan stopwords dipisah di cell ini agar tidak mengotori logika utama.
# buat kedalam utils agar menbjadi independent libarry di project asli

KAMUS_EKSPANSI = {
    # ── AI & DEEP LEARNING ─────────────────────────────────────────────
    "deep learning": ["neural network", "cnn", "rnn", "lstm", "gru", "transformer", "bert", "gpt", "autoencoder", "gan", "feedforward", "backpropagation", "jaringan saraf tiruan", "jst", "deep neural network", "dnn"],
    "machine learning": ["pembelajaran mesin", "svm", "support vector machine", "random forest", "decision tree", "naive bayes", "knn", "k-nearest neighbor", "gradient boosting", "xgboost", "regresi", "prediksi", "supervised learning", "unsupervised learning", "reinforcement learning"],
    "kecerdasan buatan": ["artificial intelligence", "ai", "sistem pakar", "expert system", "logika fuzzy", "fuzzy logic", "agen cerdas", "intelligent agent"],
    "transfer learning": ["fine tuning", "pretrained model", "feature extraction", "domain adaptation", "model reuse"],
    "generative ai": ["large language model", "llm", "diffusion model", "stable diffusion", "text to image", "prompt engineering", "rag", "retrieval augmented generation"],
    "optimasi model": ["hyperparameter tuning", "learning rate", "dropout", "regularisasi", "batch normalization", "early stopping", "overfitting", "underfitting", "cross validation"],

    # ── COMPUTER VISION ────────────────────────────────────────────────
    "klasifikasi gambar": ["image classification", "image recognition", "citra digital", "computer vision", "resnet", "vgg", "efficientnet", "inception", "mobilenet"],
    "deteksi objek": ["object detection", "yolo", "rcnn", "faster rcnn", "ssd", "feature pyramid network", "bounding box", "anchor box", "nms", "non maximum suppression"],
    "segmentasi": ["image segmentation", "semantic segmentation", "instance segmentation", "unet", "mask rcnn", "panoptic segmentation", "pixel classification"],
    "pengolahan citra": ["image processing", "opencv", "preprocessing citra", "augmentasi data", "thresholding", "edge detection", "morfologi citra", "histogram equalization", "noise reduction"],
    "pengenalan wajah": ["face recognition", "face detection", "facial landmark", "facenet", "deepface", "arcface", "liveness detection", "biometrik"],
    "ocr": ["optical character recognition", "text detection", "document analysis", "tesseract", "crnn", "handwriting recognition", "document digitization"],

    # ── NLP & TEXT ─────────────────────────────────────────────────────
    "pemrosesan bahasa alami": ["natural language processing", "nlp", "text mining", "text analysis", "corpus", "tokenisasi", "tokenization", "stemming", "lemmatization", "pos tagging"],
    "analisis sentimen": ["sentiment analysis", "opinion mining", "klasifikasi teks", "text classification", "polaritas", "emosi teks", "ulasan", "review analysis"],
    "information retrieval": ["pencarian informasi", "search engine", "temu balik informasi", "tf idf", "bm25", "inverted index", "query expansion", "relevance ranking"],
    "word embedding": ["word2vec", "fasttext", "glove", "embedding", "representasi vektor", "semantic similarity", "cosine similarity", "dense retrieval"],
    "chatbot": ["conversational ai", "dialog system", "intent recognition", "entity extraction", "ner", "named entity recognition", "slot filling", "rasa", "langchain"],
    "rangkuman teks": ["text summarization", "extractive summarization", "abstractive summarization", "keyword extraction", "keybert", "topic modeling", "lda", "bertopic"],

    # ── DATA SCIENCE ───────────────────────────────────────────────────
    "analisis data": ["data analysis", "statistik", "visualisasi data", "exploratory data analysis", "eda", "pandas", "numpy", "scipy", "deskriptif", "inferensial"],
    "data mining": ["penggalian data", "association rule", "apriori", "fp growth", "clustering", "k means", "dbscan", "hierarchical clustering", "anomaly detection"],
    "big data": ["hadoop", "spark", "apache kafka", "mapreduce", "data lake", "data warehouse", "batch processing", "stream processing", "etl"],
    "prediksi": ["forecasting", "time series", "arima", "lstm forecasting", "regresi linier", "regresi logistik", "prediksi permintaan", "prophet"],
    "preprocessing": ["data cleaning", "missing value", "outlier", "normalisasi", "standarisasi", "feature engineering", "feature selection", "imputasi", "one hot encoding", "label encoding"],

    # ── WEB & MOBILE ───────────────────────────────────────────────────
    "pengembangan web": ["web development", "frontend", "backend", "fullstack", "rest api", "graphql", "react", "vue", "angular", "next.js", "typescript"],
    "pengembangan mobile": ["mobile development", "android", "ios", "flutter", "react native", "kotlin", "swift", "progressive web app", "pwa", "cross platform"],
    "ui ux": ["user interface", "user experience", "prototyping", "wireframe", "figma", "usability testing", "heuristic evaluation", "accessibility", "desain responsif"],
    "e-commerce": ["toko online", "marketplace", "payment gateway", "keranjang belanja", "inventory management", "midtrans", "xendit", "order management"],
    "web scraping": ["crawling", "selenium", "beautifulsoup", "scrapy", "puppeteer", "data extraction", "parsing html"],

    # ── DATABASE & CLOUD ───────────────────────────────────────────────
    "basis data": ["database", "sql", "mysql", "postgresql", "sqlite", "nosql", "mongodb", "redis", "elasticsearch", "query optimization", "normalisasi database"],
    "cloud computing": ["komputasi awan", "aws", "gcp", "azure", "iaas", "paas", "saas", "serverless", "cloud storage", "cloud deployment"],
    "devops": ["ci cd", "docker", "kubernetes", "container", "pipeline", "github actions", "jenkins", "terraform", "infrastructure as code", "monitoring"],
    "microservices": ["arsitektur layanan", "api gateway", "service mesh", "message broker", "rabbitmq", "load balancing", "scalability"],

    # ── KEAMANAN & JARINGAN ────────────────────────────────────────────
    "keamanan siber": ["cybersecurity", "network security", "penetration testing", "vulnerability assessment", "enkripsi", "kriptografi", "ssl tls", "firewall", "ids ips"],
    "kriptografi": ["encryption", "rsa", "aes", "hash function", "sha", "digital signature", "public key", "private key", "blockchain", "zero knowledge proof"],
    "jaringan komputer": ["computer network", "tcp ip", "protokol", "routing", "switching", "lan wan", "vpn", "dns", "dhcp", "wireless network", "packet analysis"],
    "iot": ["internet of things", "sensor", "mikrokontroler", "arduino", "raspberry pi", "mqtt", "embedded system", "smart device", "edge computing", "firmware"],

    # ── SOFTWARE ENGINEERING ───────────────────────────────────────────
    "rekayasa perangkat lunak": ["software engineering", "sdlc", "agile", "scrum", "kanban", "waterfall", "software design", "software testing", "quality assurance"],
    "pengujian perangkat lunak": ["software testing", "unit testing", "integration testing", "black box", "white box", "regression testing", "test case", "selenium", "automated testing"],
    "sistem informasi": ["information system", "erp", "crm", "hris", "decision support system", "dss", "management information system", "enterprise system"],
    "pemodelan sistem": ["uml", "use case", "class diagram", "sequence diagram", "activity diagram", "entity relationship diagram", "erd", "data flow diagram", "dfd"],

    # ── GENERAL ────────────────────────────────────────────────────────
    "klasifikasi": ["classification", "kategorisasi", "pengelompokan"],
    "gambar":       ["image", "citra", "visual", "vision"],
}

STOPWORDS = {
    'yang', 'untuk', 'pada', 'dari', 'dengan', 'dalam', 'dan', 'atau', 'ini', 'itu', 
    'sebagai', 'oleh', 'ke', 'di', 'berbasis', 'sistem', 'menggunakan', 'terhadap',
    'analisis', 'penerapan', 'implementasi', 'studi', 'kasus', 'metode', 'of', 'and', 'for', 'the'
}

In [15]:
# Cell 3: Modul 1 - Pemrosesan Teks (Preprocessing)
# Memisahkan fungsi-fungsi yang bertugas merapikan dan mengekspansi teks.

class PreprocessingPipeline:
    @staticmethod
    def ekspansi_query(teks, kamus=KAMUS_EKSPANSI):
        teks_lower = teks.lower()
        teks_ekspansi = teks_lower
        sorted_keys = sorted(kamus.keys(), key=len, reverse=True)
        for frasa in sorted_keys:
            if frasa in teks_lower:
                teks_ekspansi += " " + " ".join(kamus[frasa])
        return teks_ekspansi

    @staticmethod
    def tokenize_ngram(teks, n=2):
        tokens = re.findall(r'\b[a-z0-9]{2,}\b', teks.lower()) 
        tokens_bersih = [t for t in tokens if t not in STOPWORDS]
        bigrams = ["_".join(g) for g in ngrams(tokens_bersih, n)]
        return tokens_bersih + bigrams

    @staticmethod
    def buat_teks_terbobot(dosen):
        keahlian = str(dosen.get('BIDANG_KEAHLIAN', ''))
        bimbing = str(dosen.get('judul bimbing', ''))
        uji = str(dosen.get('judul uji', ''))
        jurnal = str(dosen.get('JURNAL', ''))
        pendidikan = str(dosen.get('RIWAYAT_PENDIDIKAN', ''))
        
        inti = f"{keahlian} " * 3 + f"{bimbing} " * 3 + f"{uji} " * 2 + f"{jurnal} "
        teks_terbobot = f"{inti} {pendidikan}"
        
        teks_normal = f"{pendidikan} {keahlian} {jurnal} {uji} {bimbing}"
        return teks_terbobot, teks_normal

In [ ]:
# Cell 4: Modul 2 - Mesin Skoring (Lexical & Semantic)
# Mengisolasi perhitungan matematis algoritma BM25 dan Cosine Similarity SBERT.

class ScoringPipeline:
    @staticmethod
    def hitung_leksikal(token_dosen, token_mhs):
        mesin_bm25 = BM25Okapi(token_dosen)
        skor_mentah = mesin_bm25.get_scores(token_mhs)
        nilai_max = max(skor_mentah) if max(skor_mentah) > 0 else 1
        return [skor / nilai_max for skor in skor_mentah]

    @staticmethod
    def hitung_semantik(teks_dosen_list, teks_mhs):
        vektor_mhs = model_sbert.encode([teks_mhs])
        vektor_dosen = model_sbert.encode(teks_dosen_list)
        return cosine_similarity(vektor_mhs, vektor_dosen)[0] #type:ignore

In [ ]:
# Cell 5: Modul 3 - Orkestrator Utama & XAI (Explainable AI)
# Merangkai proses dari Modul 1 dan 2, mencari irisan kata/frasa, lalu memvisualisasikan grafik komparasi.
def jalankan_pipeline(data_dosen, judul_mhs, abstrak_mhs, k_rank, bobot_lex, bobot_sem):
    # --- TAHAP 1: PREPROCESSING ---
    teks_mhs_raw = judul_mhs + " " + abstrak_mhs
    teks_mhs_expand = PreprocessingPipeline.ekspansi_query(teks_mhs_raw)
    
    
    teks_bobot_list, teks_raw_list = [], []
    for dsn in data_dosen:
        tb, tr = PreprocessingPipeline.buat_teks_terbobot(dsn)
        teks_bobot_list.append(tb)
        teks_raw_list.append(tr)
        
    token_dosen = [PreprocessingPipeline.tokenize_ngram(t) for t in teks_bobot_list]
    token_mhs = PreprocessingPipeline.tokenize_ngram(teks_mhs_expand)

    # --- TAHAP 2: SCORING ---
    skor_lex = ScoringPipeline.hitung_leksikal(token_dosen, token_mhs)
    skor_sem = ScoringPipeline.hitung_semantik(teks_bobot_list, teks_mhs_expand)

    # --- TAHAP 3: PENGGABUNGAN (HYBRID) & SORTING ---
    semua_hasil = []
    for i in range(len(data_dosen)):
        skor_hybrid = (bobot_lex * skor_lex[i]) + (bobot_sem * skor_sem[i])
        semua_hasil.append({
            "indeks": i,
            "NAMA DOSEN": data_dosen[i]['NAMA'],
            "Hybrid Score": round(float(skor_hybrid), 3),
            "Lexical Score": round(float(skor_lex[i]), 3),
            "Semantic Score": round(float(skor_sem[i]), 3)
        })
        
    semua_hasil.sort(key=lambda x: x["Hybrid Score"], reverse=True)
    top_k = semua_hasil[:k_rank]

    # --- TAHAP 4: EXPLAINABLE AI (XAI) ---
    mhs_set = set(token_mhs)
    hasil_final = []
    
    for h in top_k:
        idx = h["indeks"]
        teks_dsn = teks_raw_list[idx]
        
        # Ekstrak Leksikal (Irisan)
        dsn_set = set(PreprocessingPipeline.tokenize_ngram(teks_dsn))
        irisan = list(mhs_set.intersection(dsn_set))
        kata_lex = [k.replace('_', ' ') for k in irisan]
        
        # Ekstrak Semantik (KeyBERT)
        kw = kw_model.extract_keywords(
            teks_dsn, keyphrase_ngram_range=(1, 3), stop_words=list(STOPWORDS), 
            use_maxsum=True, nr_candidates=15, top_n=3, seed_keywords=[teks_mhs_raw]
        )
        kata_sem = [k[0] for k in kw]
        
        h["Irisan Kata (Lexical)"] = ", ".join(kata_lex) if kata_lex else "-"
        h["Frasa Terkait (KeyBERT)"] = ", ".join(kata_sem) if kata_sem else "-" #type:ignore
        del h["indeks"]
        hasil_final.append(h)

    # --- TAHAP 5: VISUALISASI SKORING ---
    df_hasil = pd.DataFrame(hasil_final)
    
    return df_hasil

In [20]:
# Cell 6: Pengujian & Eksekusi
# Jalankan cell ini untuk mengetes data dan melihat hasilnya secara keseluruhan.

# Persiapan Input
judul_input = "Implementasi Deep Learning untuk Klasifikasi Gambar"
abstrak_input = "Penelitian ini menggunakan algoritma berbasis kecerdasan buatan untuk mendeteksi objek."

# Memuat Dataset
DATASET_PATH = "C:/Users/USER/Desktop/github-project/sistem_rekomendasi_clone/NLP-Base-Recommendation-System/server/data/dataset_profiles_terintegrasi.xlsx"
try:
    dataset_dosen = pd.read_excel(DATASET_PATH)
    data_dict = dataset_dosen.to_dict('records')
except FileNotFoundError:
    print("Dataset tidak ditemukan. Menggunakan data dummy sementara.")
    data_dict = [{"NAMA": "Dr. A", "BIDANG_KEAHLIAN": "computer vision, neural network", "RIWAYAT_PENDIDIKAN": "S3 AI"},
                 {"NAMA": "Bpk. B", "BIDANG_KEAHLIAN": "sistem informasi, database", "RIWAYAT_PENDIDIKAN": "S2 SI"}]

# Eksekusi Pipeline Utama
df_rekomendasi = jalankan_pipeline(
    data_dosen=data_dict,  
    judul_mhs=judul_input,
    abstrak_mhs=abstrak_input,
    k_rank=5,                 
    bobot_lex=0.3,      
    bobot_sem=0.7      
)

# Tampilkan Tabel Output Akhir
pd.set_option('display.max_colwidth', None)
# print("\n📝 VISUALISASI PIPELINE 4: TABEL REKOMENDASI & XAI")
display(df_rekomendasi)

,NAMA DOSEN,Hybrid Score,Lexical Score,Semantic Score,Irisan Kata (Lexical),Frasa Terkait (KeyBERT)
0,"Agung Riyadi, S.Si., M.Kom",0.726,1.000,0.608,"system, visual, jaringan, classification, algoritma, klasifikasi, learning, ai, gambar, neural network, image, cnn, intelligence, network, digital, klasifikasi gambar, neural, artificial intelligence, network cnn, box, artificial","inventori web klasifikasi, algoritma neural network, pengembangan game edukasi"
1,"Sukma Evadini, S.T., M.Kom",0.642,0.639,0.643,"expert, deep learning, intelligence, algoritma, learning, recognition, deep, artificial, system, artificial intelligence, visual, computer, expert system","kesehatan algoritma frequent, indonesia ilmu komputer, deep learning models"
2,"Ahmadi Irmansyah Lubis, S.kom., M.kom.",0.628,0.815,0.548,"system, citra, computer, jaringan, classification, cerdas, algoritma, learning, ai, image recognition, image classification, image, intelligence, digital, lstm, artificial intelligence, yolo, deep learning, recognition, deep, artificial, detection","system machine learning, analysis accuracy improvement, intelligence decision support"
3,Fendra Dwi R,0.628,0.963,0.484,"image, deep learning, object, klasifikasi, fuzzy, learning, digital, objek, network, maximum, fuzzy logic, deep, artificial, system, neural, citra, logic, neural network","sensing gis geoinformatics, analysis urban built, 2012 pemetaan banjir"
4,"Alena Uperiati, S.T, M.Cs",0.613,0.708,0.572,"vision, system, classification, algoritma, cerdas, klasifikasi, learning, ai, gambar, neural network, image, cnn, intelligence, network, digital, klasifikasi gambar, neural, artificial intelligence, recognition, artificial","batam melalui algoritma, komputer machine learning, mendukung digitalisasi umkm"
